In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("..")

In [3]:
from IPython.display import clear_output
from src.dataset_loaders_new import get_samplers, get_indices
from src.utils import get_pca_models
from src import utils
from src.train import train_discrete
import wandb
from torch.utils.data import TensorDataset, DataLoader
import yaml
import numpy as np
import random
import pickle
import gc

## 1. Parameters.

Possible ```DATASET_NAME``` values are: ```twitter```, ```wiki-gigaword```, ```bone_marrow```

In [4]:
FUSED_DIM    = 30
SOURCE_DIM   = 100  #38 (bone_marrow)
TARGET_DIM   = 50  #50
MAX_ITERS    = 700

In [5]:

DATASET_NAME = 'twitter'
METHOD_NAME  = 'FlowGW'
DEVICE       = 'cpu'

ALPHA        = 0.9
SEED         = 43
COST_DISCRETE= 'cosine'

# Just a security check to not alter the NN input and output dims.
# The second condition makes regular OT for fused dim higher than 25

if DATASET_NAME != 'bone_marrow':
    FUSED_DIM = 0
elif FUSED_DIM > 25:
    SOURCE_DIM = FUSED_DIM
    TARGET_DIM = FUSED_DIM


config = {'dataset':dict(DATASET_NAME     = DATASET_NAME,
                         DEVICE           = DEVICE,
                         SOURCE_DIM       = SOURCE_DIM,
                         TARGET_DIM       = TARGET_DIM,
                         FUSED_DIM        = FUSED_DIM,
                         N_MAX_SAMPLES    = 400000, #set to 6667 to get N_train=3K
                         N_TRAIN_SAMPLES  = 6000, #We used 6000 for the others
                         N_TEST_SAMPLES   = 256,
                         N_EVAL           = 4,
                         ALPHA            = ALPHA, 
                         SEED             = SEED,
                         TRAIN_TYPE       = 'discrete',
                         NORMALIZE_VECS   = False
                          ),
          
          'training':dict(METHOD_NAME          = METHOD_NAME,
                          MAX_ITERS            = MAX_ITERS,
                          COST_DISCRETE        = COST_DISCRETE,
                          ),

          #'model_specific':dict(HIDDEN_SIZES_MLP = [512, 256, 256],
          #                      EPS_FIT          = 0.01,
          #                      EPS_REG          = 0.001,
          #                      LAMBDA           = 1,
          #                      MOVER_LR         = 1e-4
          #                     ),
          
          'model_specific':dict(HIDDEN_SIZES_MLP = [1024, 1024, 1024, 1024],
                                EPSILON          = 1e-3,
                                N_FREQ           = 128,
                                MOVER_LR         = 1e-4
                               )}



## 2. Loading dataset.

In [6]:
dataset_path = '../datasets'
sys.path.append(dataset_path)

source_vectors, target_vectors, random_indices_train, random_indices_test = get_indices(dataset_path, config)
source_vectors, target_vectors, trainloader, testloader = get_samplers(source_vectors, target_vectors, random_indices_train, random_indices_test, config)

print(source_vectors.shape)
print(target_vectors.shape)
#print(random_indices_train[:200000][:10])
#print(random_indices_test)#[200000:][:10])

Loading model twitter_100 to source...
Loading model twitter_50 to target...
Source pairs...
3000
Target pairs...
3001
torch.Size([400000, 100])
torch.Size([400000, 50])


## 3. Training.

In [ ]:
import os
#os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'

n_repeats = 10
wandb_report = True
project_name = f'{METHOD_NAME}_new_{DATASET_NAME}_{SOURCE_DIM}->{TARGET_DIM}_6K_{n_repeats}rep'

metrics_names = ['Top@1', 'Top@5', 'Top@10', 'cossim_gt', 'inner_gw', 'foscttm']
_, _, _, random_indices_test_fixed = get_indices(dataset_path, config)      

alpha_values = [0.0, 0.1, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9][::-1]
#alpha_values = [0.0][::-1]

metrics_out = {str(np.round(alpha, 1)):[] for alpha in alpha_values}

for ALPHA in alpha_values:
    
    config['dataset']['ALPHA'] = ALPHA 
    
    print('================================')
    print(f'Experiment for ALPHA={ALPHA}')
    print('================================')
    
    for ix in range(n_repeats):
        
        if wandb_report:
            exp_name = f'ALPHA_{np.round(ALPHA, 1)}_repeat_{ix}'
            wandb.init(name=exp_name, config=config, project=project_name)
            
        SEED = random.randint(0, 10000)
        config['dataset']['SEED'] = SEED
        print('Seed: ', SEED)
        
        source_vectors, target_vectors, random_indices_train, _ = get_indices(dataset_path, config)            
        source_vectors, target_vectors, train_sampler, test_sampler = get_samplers(source_vectors, target_vectors, random_indices_train, random_indices_test_fixed, config)

        #pca_models = get_pca_models(source_vectors, target_vectors)
        
        trained_class, metrics_dict = train_discrete(train_sampler, test_sampler, 
                                                     metrics_names, target_vectors,
                                                     config,
                                                     wandb_report=wandb_report,
                                                     axis_lims=None, report_every=100)
        
        metrics_out[str(np.round(ALPHA, 1))].append(metrics_dict)

        with open(f'results_discrete/{project_name}.pkl', 'wb') as f:
            pickle.dump(metrics_out, f)

        del trained_class, metrics_dict, source_vectors, target_vectors, random_indices_train, train_sampler, test_sampler
        gc.collect()

Loading model twitter_100 to source...
Loading model twitter_50 to target...
Experiment for ALPHA=0.9


wandb: Currently logged in as: xavier13091994 (entropic_gw). Use `wandb login --relogin` to force relogin


Seed:  7645
Loading model twitter_100 to source...
Loading model twitter_50 to target...
Source pairs...
3000
Target pairs...
3001


2024-10-01 14:55:55.150675: W external/xla/xla/service/gpu/nvptx_compiler.cc:765] The NVIDIA driver's CUDA version is 12.3 which is older than the ptxas CUDA version (12.5.40). Because the driver is older than the ptxas version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.


  0%|          | 0/700 [00:00<?, ?it/s]

...

/trinity/home/xavier.aramayo/miniconda3/envs/jax/lib/python3.10/site-packages/jax/_src/core.py:678: FutureWarning: unhashable type: <class 'jax._src.interpreters.partial_eval.DynamicJaxprTracer'>. Attempting to hash a tracer will lead to an error in a future JAX release.
  warnings.warn(
/trinity/home/xavier.aramayo/miniconda3/envs/jax/lib/python3.10/site-packages/jax/_src/core.py:678: FutureWarning: unhashable type: <class 'jax._src.interpreters.partial_eval.DynamicJaxprTracer'>. Attempting to hash a tracer will lead to an error in a future JAX release.
  warnings.warn(
/trinity/home/xavier.aramayo/miniconda3/envs/jax/lib/python3.10/site-packages/jax/_src/core.py:678: FutureWarning: unhashable type: <class 'jax._src.interpreters.partial_eval.DynamicJaxprTracer'>. Attempting to hash a tracer will lead to an error in a future JAX release.
  warnings.warn(
/trinity/home/xavier.aramayo/miniconda3/envs/jax/lib/python3.10/site-packages/jax/_src/core.py:678: FutureWarning: unhashable type: <

test/Top@1,▁▄▅▆▇▇█
test/Top@10,▁▅▆▇▇██
test/Top@5,▁▄▆▆▇██
test/cossim_gt,▁▄▆▆▇██
test/foscttm,█▅▄▃▂▂▁
test/inner_gw,█▅▄▁▁▂▃
train/Top@1,▁▄▅▆▇▇█
train/Top@10,▁▅▆▆▇▇█
train/Top@5,▁▄▆▆▇▇█
train/cossim_gt,▁▅▆█▇▆▅
train/foscttm,▆▁▁█▃▆█


Seed:  6706
Loading model twitter_100 to source...
Loading model twitter_50 to target...
Source pairs...
3000
Target pairs...
3001


  0%|          | 0/700 [00:00<?, ?it/s]

/trinity/home/xavier.aramayo/miniconda3/envs/jax/lib/python3.10/site-packages/jax/_src/core.py:678: FutureWarning: unhashable type: <class 'jax._src.interpreters.partial_eval.DynamicJaxprTracer'>. Attempting to hash a tracer will lead to an error in a future JAX release.
  warnings.warn(
/trinity/home/xavier.aramayo/miniconda3/envs/jax/lib/python3.10/site-packages/jax/_src/core.py:678: FutureWarning: unhashable type: <class 'jax._src.interpreters.partial_eval.DynamicJaxprTracer'>. Attempting to hash a tracer will lead to an error in a future JAX release.
  warnings.warn(
/trinity/home/xavier.aramayo/miniconda3/envs/jax/lib/python3.10/site-packages/jax/_src/core.py:678: FutureWarning: unhashable type: <class 'jax._src.interpreters.partial_eval.DynamicJaxprTracer'>. Attempting to hash a tracer will lead to an error in a future JAX release.
  warnings.warn(
/trinity/home/xavier.aramayo/miniconda3/envs/jax/lib/python3.10/site-packages/jax/_src/core.py:678: FutureWarning: unhashable type: <